# Challenge 1: datos y análisis exploratorio

Tu primera responsabilidad es preparar una fuente de datos confiable para el proyecto de clasificación de vinos.

**Entrada:** `data/raw/wine.csv`  
**Salidas:** `data/processed/train.csv`, `data/processed/test.csv` y `artifacts/data_contract.json`

> El conjunto de test se crea aquí, pero no debe explorarse ni utilizarse después del split.


## Objetivos

- cargar y validar datos externos;
- identificar problemas básicos de calidad;
- crear un split reproducible y estratificado;
- realizar EDA únicamente sobre entrenamiento;
- guardar datos y metadatos para la siguiente etapa.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga y verificación inicial


In [ ]:
DATA_PATH = RAW_DATA_DIR / "wine.csv"
if not DATA_PATH.exists() and (PROJECT_DIR / "data" / "wine.csv").exists():
    DATA_PATH = PROJECT_DIR / "data" / "wine.csv"

df = pd.read_csv(DATA_PATH)
print("Dimensiones del dataset:", df.shape)
df.head()


In [ ]:
print("=== INFORMACIÓN GENERAL Y TIPOS DE DATOS ===")
print(df.info())

print("\n=== VALORES FALTANTES POR COLUMNA ===")
print(df.isnull().sum())

print(f"\nFilas duplicadas: {df.duplicated().sum()}")

print("\n=== DISTRIBUCIÓN DE LA CLASE TARGET ===")
print(df["target"].value_counts(normalize=True).apply(lambda x: f"{x:.2%}"))
print(df["target"].value_counts())


### Diagnóstico de calidad del dataset

El conjunto consta de 178 registros y 13 variables fisicoquímicas continuas más la variable objetivo categórica `target`. No se detectan valores nulos ni duplicados. La distribución de etiquetas muestra tres clases moderadamente balanceadas: Clase 1 ($39.89\%$, $n=71$), Clase 0 ($33.15\%$, $n=59$) y Clase 2 ($26.97\%$, $n=48$). Se fija una partición estratificada 80/20 (`stratify=df['target']`) para garantizar que las frecuencias relativas de las clases se preserven idénticas en entrenamiento ($n=142$) y prueba ($n=36$).


## 2. Split reproducible y estratificado


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],
    random_state=RANDOM_STATE
)


In [ ]:
assert len(train_df) + len(test_df) == len(df)
assert set(train_df.index).isdisjoint(set(test_df.index))
print("Train:", train_df.shape)
print("Test:", test_df.shape)

## 3. Análisis exploratorio sobre entrenamiento


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=train_df, x="target", ax=axes[0], palette="viridis")
axes[0].set_title("Distribución de Clases (Train)")
axes[0].set_xlabel("Target (Clase de Vino)")
axes[0].set_ylabel("Frecuencia")

# 2. Boxplot de Flavanoids por clase
sns.boxplot(data=train_df, x="target", y="flavanoids", ax=axes[1], palette="Set2")
axes[1].set_title("Flavanoides por Clase")
axes[1].set_xlabel("Target")
axes[1].set_ylabel("Flavanoids")

# 3. Dispersión Alcohol vs Proline por clase
sns.scatterplot(
    data=train_df,
    x="alcohol",
    y="proline",
    hue="target",
    palette="deep",
    ax=axes[2]
)
axes[2].set_title("Alcohol vs Proline")
axes[2].set_xlabel("Alcohol")
axes[2].set_ylabel("Proline")

plt.tight_layout()
plt.show()

# 4. Matriz de correlación
plt.figure(figsize=(10, 8))
corr = train_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Matriz de Correlación (Train)")
plt.show()


### Conclusiones del análisis exploratorio

Las variables `flavanoids` y `total_phenols` exhiben alto poder de separación ($r \approx -0.85$ con respecto a la etiqueta), discriminando la clase 0 (alta concentración), clase 1 (media) y clase 2 (baja). La disparidad de escalas es notable (`proline` en $[200, 1600]$ frente a `nonflavanoid_phenols` en $[0.1, 0.7]$), por lo que se requiere estandarización con `StandardScaler` en estimadores sensibles a distancias euclidianas o problemas de acondicionamiento de matriz en optimización por gradiente. En el plano bivariado, la interacción entre `alcohol` y `proline` genera una frontera lineal bien condicionada para la clase 0.


## 4. Guardar datos procesados y contrato


In [ ]:
train_df_reset = train_df.reset_index(drop=True)
test_df_reset = test_df.reset_index(drop=True)

train_df_reset.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df_reset.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

feature_cols = [c for c in df.columns if c != "target"]
data_contract = {
    "target": "target",
    "features": feature_cols,
    "random_state": RANDOM_STATE,
    "test_size": 0.2,
    "n_train_samples": len(train_df_reset),
    "n_test_samples": len(test_df_reset)
}

(ARTIFACTS_DIR / "data_contract.json").write_text(
    json.dumps(data_contract, indent=2),
    encoding="utf-8"
)
print("✓ Salidas guardadas: train.csv, test.csv y data_contract.json")


In [ ]:
assert (PROCESSED_DATA_DIR / "train.csv").exists()
assert (PROCESSED_DATA_DIR / "test.csv").exists()
assert (ARTIFACTS_DIR / "data_contract.json").exists()
print("Challenge 1 completado.")

## Checklist

Antes de pasar al siguiente notebook confirma:

- [x] `train.csv` y `test.csv` existen en `data/processed/`.
- [x] El split se hizo antes de cualquier cálculo descriptivo.
- [x] El EDA usó únicamente `train_df`.
- [x] `artifacts/data_contract.json` contiene la lista de features y la columna target.
